### How to Upload a File to Colab:

**Option 1: Using the Files Tab (Recommended for small files)**

1.  **Click the 'Files' icon** on the left sidebar (it looks like a folder).
2.  **Click the 'Upload to session storage' icon** (it looks like an arrow pointing upwards) within the Files pane.
3.  **Select your `online_retail.csv` file** from your local computer and click 'Open'.

The file will be uploaded to the root directory of your Colab session, making it accessible by its filename directly (e.g., `'online_retail.csv'`).

**Option 2: Using Code (for larger files or programmatic uploads)**

You can run the following Python code to get an upload prompt:

```python
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')
```

After running this, a button will appear that says 'Choose Files'. Click it, select your `online_retail.csv` file, and it will be uploaded to the session.

In [2]:
import pandas as pd
df = pd.read_csv('online_retail.csv')
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [3]:
import sqlite3
conn = sqlite3.connect('retail.db')
df.to_sql('online_retail', conn, if_exists='replace', index=False)

541909

In [3]:
query = "SELECT * FROM online_retail LIMIT 10;"

In [4]:
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


,0
InvoiceNo,0
StockCode,0
Description,1454
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,135080
Country,0


In [5]:
query = """SELECT
CustomerId, MAX(InvoiceDate) as LastPurchaseDate,
COUNT(DISTINCT InvoiceNo) as Frequency,
SUM(Quantity * UnitPrice)
FROM online_retail
GROUP BY CustomerId
"""

rfm_df = pd.read_sql_query(query, conn)
rfm_df.head()

,CustomerID,LastPurchaseDate,Frequency,SUM(Quantity * UnitPrice)
0,NaN,2011-12-09 10:26:00,3710,1447682.12
1,12346.0,2011-01-18 10:17:00,2,0.00
2,12347.0,2011-12-07 15:52:00,7,4310.00
3,12348.0,2011-09-25 13:13:00,4,1797.24
4,12349.0,2011-11-21 09:51:00,1,1757.55


In [6]:
query = """SELECT MAX(InvoiceDate)
FROM online_retail"""
max_date = pd.read_sql_query(query, conn)
max_date

,MAX(InvoiceDate)
0,2011-12-09 12:50:00


In [7]:
# getting the actual date from DataFrame
ref_date = max_date.iloc[0, 0]
ref_date

'2011-12-09 12:50:00'

In [8]:
# the f before f string maks sure that ref_date gets used
query = f"""
SELECT
    CustomerID,
    julianday('{ref_date}') - julianday(MAX(InvoiceDate)) AS Recency
FROM online_retail
GROUP BY CustomerID
"""

recency_df = pd.read_sql_query(query, conn)
recency_df

,CustomerID,Recency
0,NaN,0.100000
1,12346.0,325.106250
2,12347.0,1.873611
3,12348.0,74.984028
4,12349.0,18.124306
...,...,...
4368,18280.0,277.123611
4369,18281.0,180.081250
4370,18282.0,7.046528
4371,18283.0,3.033333


In [9]:
query = f"""SELECT
    CustomerID,
    julianday('{ref_date}') - julianday(MAX(InvoiceDate)) AS Recency,
    COUNT(DISTINCT InvoiceNo) AS Frequency,
    SUM(Quantity * UnitPrice) AS Monetary
FROM online_retail
GROUP BY CustomerID
"""
some_df = pd.read_sql_query(query, conn)
some_df

,CustomerID,Recency,Frequency,Monetary
0,NaN,0.100000,3710,1447682.12
1,12346.0,325.106250,2,0.00
2,12347.0,1.873611,7,4310.00
3,12348.0,74.984028,4,1797.24
4,12349.0,18.124306,1,1757.55
...,...,...,...,...
4368,18280.0,277.123611,1,180.60
4369,18281.0,180.081250,1,80.82
4370,18282.0,7.046528,3,176.60
4371,18283.0,3.033333,16,2094.88


In [10]:
# labels are reversed as most recent customers has to have score 4
some_df['R_score'] = pd.qcut(some_df['Recency'], q = 4, labels = [4,3,2,1])
some_df

,CustomerID,Recency,Frequency,Monetary,R_score
0,NaN,0.100000,3710,1447682.12,4
1,12346.0,325.106250,2,0.00,1
2,12347.0,1.873611,7,4310.00,4
3,12348.0,74.984028,4,1797.24,2
4,12349.0,18.124306,1,1757.55,3
...,...,...,...,...,...
4368,18280.0,277.123611,1,180.60,1
4369,18281.0,180.081250,1,80.82,1
4370,18282.0,7.046528,3,176.60,4
4371,18283.0,3.033333,16,2094.88,4


In [11]:
# Problem: many customers share the same Frequency value
# duplicates='drop' told pandas "if edges collide, just merge those into one bin instead of erroring out.
# some_df['F_score'] = pd.qcut(some_df['Frequency'], q = 4, labels = [1,2,3,4], duplicates = 'drop')
some_df['F_score'] = pd.qcut(some_df['Frequency'], q = 4, labels = [1,2,3], duplicates = 'drop')
some_df

,CustomerID,Recency,Frequency,Monetary,R_score,F_score
0,NaN,0.100000,3710,1447682.12,4,3
1,12346.0,325.106250,2,0.00,1,1
2,12347.0,1.873611,7,4310.00,4,3
3,12348.0,74.984028,4,1797.24,2,2
4,12349.0,18.124306,1,1757.55,3,1
...,...,...,...,...,...,...
4368,18280.0,277.123611,1,180.60,1,1
4369,18281.0,180.081250,1,80.82,1,1
4370,18282.0,7.046528,3,176.60,4,1
4371,18283.0,3.033333,16,2094.88,4,3


In [12]:
some_df['M_score'] = pd.qcut(some_df['Monetary'], q = 4, labels = [1,2,3,4])
some_df

,CustomerID,Recency,Frequency,Monetary,R_score,F_score,M_score
0,NaN,0.100000,3710,1447682.12,4,3,4
1,12346.0,325.106250,2,0.00,1,1,1
2,12347.0,1.873611,7,4310.00,4,3,4
3,12348.0,74.984028,4,1797.24,2,2,4
4,12349.0,18.124306,1,1757.55,3,1,4
...,...,...,...,...,...,...,...
4368,18280.0,277.123611,1,180.60,1,1,1
4369,18281.0,180.081250,1,80.82,1,1,1
4370,18282.0,7.046528,3,176.60,4,1,1
4371,18283.0,3.033333,16,2094.88,4,3,4


In [13]:
def get_label(row):
    if row['R_score'] >= 3 and row['F_score'] >= 3 and row['M_score'] >= 3:
        return "Best Customers"
    elif row['R_score'] >= 3 and row['F_score'] <= 1 and row['M_score'] <= 2:
        return "New/Promising"
    elif row['R_score'] <= 2 and row['F_score'] >= 2 and row['M_score'] >= 2:
        return "At Risk"
    elif row['R_score'] <= 2 and row['F_score'] <= 1 and row['M_score'] <= 2:
        return "Lost/Churned"
    else:
        return "Regular"

some_df['Segment'] = some_df.apply(get_label, axis=1)
some_df

,CustomerID,Recency,Frequency,Monetary,R_score,F_score,M_score,Segment
0,NaN,0.100000,3710,1447682.12,4,3,4,Best Customers
1,12346.0,325.106250,2,0.00,1,1,1,Lost/Churned
2,12347.0,1.873611,7,4310.00,4,3,4,Best Customers
3,12348.0,74.984028,4,1797.24,2,2,4,At Risk
4,12349.0,18.124306,1,1757.55,3,1,4,Regular
...,...,...,...,...,...,...,...,...
4368,18280.0,277.123611,1,180.60,1,1,1,Lost/Churned
4369,18281.0,180.081250,1,80.82,1,1,1,Lost/Churned
4370,18282.0,7.046528,3,176.60,4,1,1,New/Promising
4371,18283.0,3.033333,16,2094.88,4,3,4,Best Customers


In [14]:
some_df['Segment'].value_counts()

,count
Segment,
Lost/Churned,1396
Regular,1065
Best Customers,885
New/Promising,611
At Risk,416


In [15]:
# csv export - download to computer
some_df.to_csv('rfm_results.csv', index=False)
from google.colab import files
files.download('rfm_results.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
# Total revenue and total customers
total_revenue = some_df['Monetary'].sum()
total_customers = len(some_df)

# Revenue and customer count per segment
segment_summary = some_df.groupby('Segment').agg(
    customer_count=('CustomerID', 'count'),
    total_revenue=('Monetary', 'sum')
)

# Add percentage columns
segment_summary['pct_of_customers'] = (segment_summary['customer_count'] / total_customers * 100).round(1)
segment_summary['pct_of_revenue'] = (segment_summary['total_revenue'] / total_revenue * 100).round(1)

segment_summary

,customer_count,total_revenue,pct_of_customers,pct_of_revenue
Segment,,,,
At Risk,416,765713.010,9.5,7.9
Best Customers,884,7069841.460,20.2,72.5
Lost/Churned,1396,377237.832,31.9,3.9
New/Promising,611,193738.860,14.0,2.0
Regular,1065,1341216.772,24.4,13.8
